In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

NOISE_AMP = .0005
NOISE_OFFSET = 0.01

WEIGHTS = torch.tensor([1.0, 1.0, 5.0]).to(DEVICE)


def weighted_mse_loss(predictions, targets):
    return (((predictions - targets) ** 2)*WEIGHTS).mean()


import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity  
        out = self.relu(out)
        return out

class CRDS_ResYOLO(nn.Module):
    def __init__(self):
        super(CRDS_ResYOLO, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        # Residual Layers
        self.layer1 = self._make_layer(32, 64, stride=(2, 2))  # Replaces MaxPool2d(2)
        self.layer2 = self._make_layer(64, 128, stride=(1, 2)) # Replaces MaxPool2d((1, 2))
        
        self.pool = nn.AdaptiveAvgPool2d((1, 10)) 

        self.regressor = nn.Sequential(
            nn.Linear(1280, 512),
            nn.GELU(),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 16 * 3)
        )

    def _make_layer(self, in_channels, out_channels, stride):
        downsample = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        return ResidualBlock(in_channels, out_channels, stride, downsample)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x)
        
        x = torch.flatten(x, 1)
        x = self.regressor(x)
        
        
        return x.view(-1, 16, 4)

class BlobDataset(Dataset):
    def __init__(self, data_path):
        
        data = joblib.load(data_path)
        self.blobs = data['blobs']
       
        self.centers = data["center"]
        self.intensities = data["intensity"]
        self.classify = data["classify"]
    
    def __len__(self):
        return len(self.blobs)

    def __getitem__(self, idx):

        scale = np.random.randint(300,1600)
        BACKGROUND_LEVEL = 100

        OGimg = self.blobs[idx]

        img = OGimg * scale
    
        row_offset = np.random.uniform(-4, 2, size=(img.shape[0], 1))

        shot_noise = np.random.normal(0, 5, size=img.shape)

        col_pattern = np.random.normal(0, 1, size=(1, img.shape[1]))

        img = img + BACKGROUND_LEVEL + row_offset + shot_noise + col_pattern

        
        img = torch.from_numpy(img).float().view(1, 30, 640)
        
        c = torch.from_numpy(self.centers[idx]).float()

        integ = self.intensities[idx] * scale

        i = torch.from_numpy(integ).float().unsqueeze(-1)

        cl = torch.from_numpy(self.classify[idx]).float()

        target = torch.cat([c, i, cl], dim=1) 
        
        return img, target

if __name__ == "__main__":
    train_ds = BlobDataset("New_MultiBlob_TRA_DAT.joblib")
    val_ds = BlobDataset("New_MultiBlob_VAL_DAT.joblib")

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = CRDS_ResYOLO().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    best_val_loss = float('inf')

    for epoch in range(50):
        model.train()
        train_loss = 0.0
        for img, targ in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            img, targ = img.to(DEVICE), targ.to(DEVICE)
            

            optimizer.zero_grad()
            preds = model(img)
            

            loss = weighted_mse_loss(preds, targ)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for v_img, v_targ in val_loader:
                v_img, v_targ = v_img.to(DEVICE), v_targ.to(DEVICE)
                v_preds = model(v_img)
                val_loss += weighted_mse_loss(v_preds, v_targ).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        scheduler.step(avg_val)
        
        print(f"Summary: Train {avg_train:.6f} | Val {avg_val:.6f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), "best_model_v1.pt")
            print(f"--> Saved New Best Model\n")